In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jeber81_task01_braintumour_path = kagglehub.dataset_download('jeber81/task01-braintumour')
jeber81_brain_tumor_segmentation_path = kagglehub.notebook_output_download('jeber81/brain-tumor-segmentation')

print('Data source import complete.')


# Brain tumor 3D segmentation with MONAI
**Kaggle Version**
Remember to:
1. Select **PyTorch** as the framework.
2. Add the dataset to your Kaggle environment so it mounts at `/kaggle/input/`

In [ ]:
# Force downgrade to PyTorch 2.4.1 to restore P100 support
!pip install -U -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q monai medpy
%matplotlib inline

In [ ]:
import os
import sys
import contextlib

# 1. Set environment variables BEFORE any other imports
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['JAX_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GLOG_minloglevel'] = '3' # Suppresses C++ Google Logging

# 2. Low-level helper to catch C++ stderr noise during imports
@contextlib.contextmanager
def suppress_cpp_noise():
    """Redirects system-level stderr to devnull to catch C++ framework warnings."""
    with open(os.devnull, 'w') as devnull:
        # Save original file descriptors
        stderr_fd = sys.stderr.fileno()
        with os.fdopen(os.dup(stderr_fd), 'w') as old_stderr:
            sys.stderr.flush()
            os.dup2(devnull.fileno(), stderr_fd)
            try:
                yield
            finally:
                sys.stderr.flush()
                os.dup2(old_stderr.fileno(), stderr_fd)

# 3. Perform the "noisy" imports inside the suppressor
with suppress_cpp_noise():
    import glob
    import time
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    import warnings
    import logging
    from tqdm import tqdm

    # Initializing absl explicitly often stops the "InitializeLog" warning
    try:
        import absl.logging
        absl.logging.set_verbosity(absl.logging.ERROR)
    except:
        pass

    from monai.data import DataLoader, Dataset, decollate_batch
    from monai.losses import DiceLoss
    from monai.inferers import sliding_window_inference
    from monai.metrics import DiceMetric
    from monai.networks.nets import UNet
    from monai.transforms import (
        Activations, AsDiscrete, Compose, EnsureChannelFirstd, EnsureTyped,
        LoadImaged, MapTransform, NormalizeIntensityd, Orientationd,
        RandFlipd, RandScaleIntensityd, RandShiftIntensityd,
        RandSpatialCropd, Spacingd
    )
    from monai.utils import set_determinism
    from sklearn.model_selection import KFold
    from medpy.metric.binary import hd95

# 4. Standard Python suppressors for the rest of the session
logging.getLogger('absl').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
set_determinism(seed=0)

print("Imports completed cleanly.")

In [ ]:
# --- DIRECTORY SETUP FOR KAGGLE ---
drive_save_dir = '/kaggle/working/'
os.makedirs(drive_save_dir, exist_ok=True)

# Update this path to exactly where your files are!
data_dir = '/kaggle/input/datasets/jeber81/task01-braintumour/Task01_BrainTumour/'

print(f"[SAVE VAULT] Models will be saved to: {drive_save_dir}")
print(f"[FAST READ] Data will be loaded from: {data_dir}")

In [ ]:
class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            result.append(torch.logical_or(d[key] == 2, d[key] == 3))
            result.append(torch.logical_or(torch.logical_or(d[key] == 2, d[key] == 3), d[key] == 1))
            result.append(d[key] == 3)
            d[key] = torch.stack(result, axis=0).float()
        return d

In [ ]:
train_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    EnsureTyped(keys=["image", "label"]),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    RandSpatialCropd(keys=["image", "label"], roi_size=[64, 64, 64], random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
    RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
])

val_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    EnsureTyped(keys=["image", "label"]),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

# ---> UPDATE IS HERE: Changed *.nii.gz to *.nii <---
train_images = sorted(glob.glob(os.path.join(data_dir, "imagesTr", "*.nii")))
train_labels = sorted(glob.glob(os.path.join(data_dir, "labelsTr", "*.nii")))

data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(train_images, train_labels)]

print(f"Total training subjects ready for splitting: {len(data_dicts)}")

In [ ]:
# --- TASK 2.1: DATA VISUALIZATION ---
# Pick a random training subject to visualize
sample_data = val_transform(data_dicts[0])
img_tensor = sample_data["image"]
label_tensor = sample_data["label"]

# Select a middle slice (z-axis)
slice_idx = 45

plt.figure(figsize=(15, 5))
# Plotting the FLAIR channel (channel 0)
plt.subplot(1, 4, 1)
plt.title(f"FLAIR Image (Slice {slice_idx})")
plt.imshow(img_tensor[0, :, :, slice_idx].detach().cpu().numpy(), cmap="gray")
plt.axis("off")

# Plotting the 3 sub-regions (TC, WT, ET) we created in our Transform
titles = ["Tumor Core (TC)", "Whole Tumor (WT)", "Enhancing Tumor (ET)"]
for i in range(3):
    plt.subplot(1, 4, i+2)
    plt.title(f"{titles[i]} Mask")
    plt.imshow(label_tensor[i, :, :, slice_idx].detach().cpu().numpy(), cmap="viridis")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# --- SETUP CROSS VALIDATION & METRICS ---
kf = KFold(n_splits=5, shuffle=True, random_state=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")

post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

def safe_hd95(pred, gt):
    if pred.sum() > 0 and gt.sum() > 0:
        return hd95(pred.cpu().numpy(), gt.cpu().numpy(), voxelspacing=(1.0, 1.0, 1.0))
    return np.nan

max_epochs = 30
val_interval = 2
fold_results_dice = {}
fold_results_hd95 = {}

In [ ]:
# --- FINAL OPTIMIZED TRAINING LOOP & ASSIGNMENT REPORTING ---
total_start = time.time()

# Patch size and epochs optimized for Kaggle's 12-hour limit
patch_size = (64, 64, 64)
max_epochs = 20

# !!! IMPORTANT !!!
# --- AUTO-DETECT PREVIOUS OUTPUTS DIRECTORY ---
previous_outputs_dir = ""
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'multimodal_unet_checkpoint_fold_1.pth' in files:
        previous_outputs_dir = root
        print(f"\n[SUCCESS] Auto-detected previous checkpoints at: {previous_outputs_dir}\n")
        break

if not previous_outputs_dir:
    print("\n[CRITICAL WARNING] Could not find the checkpoint files in /kaggle/input/!\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(data_dicts)):
    print(f"\n{'='*50}\n Spinning up Fold {fold + 1}/5\n{'='*50}")

    checkpoint_name = f"multimodal_unet_checkpoint_fold_{fold+1}.pth"
    best_model_name = f"best_unet_fold_{fold+1}.pth"

    # Paths for saving (current run) and loading (previous run)
    checkpoint_path_working = os.path.join(drive_save_dir, checkpoint_name)
    checkpoint_path_input = os.path.join(previous_outputs_dir, checkpoint_name)

    train_files = [data_dicts[i] for i in train_idx]
    val_files = [data_dicts[i] for i in val_idx]

    train_ds = Dataset(data=train_files, transform=train_transform)
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)

    val_ds = Dataset(data=val_files, transform=val_transform)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

    model = UNet(
        spatial_dims=3, in_channels=4, out_channels=3,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2), num_res_units=2,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
    scaler = torch.GradScaler("cuda")

    start_epoch = 0
    best_metric = -1

    # --- CHECKPOINT RECOVERY ---
    load_path = None
    if os.path.exists(checkpoint_path_working):
        load_path = checkpoint_path_working
    elif os.path.exists(checkpoint_path_input):
        load_path = checkpoint_path_input

    if load_path:
        print(f"[*] Found existing checkpoint! Loading from {load_path}...")
        checkpoint = torch.load(load_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_metric = checkpoint.get('best_metric', -1)

        # Restore metrics for the final table if they exist
        if 'fold_dice' in checkpoint: fold_results_dice[fold+1] = checkpoint['fold_dice']
        if 'fold_hd95' in checkpoint: fold_results_hd95[fold+1] = checkpoint['fold_hd95']

        print(f"[*] Resuming training from Epoch {start_epoch + 1}...")

    def inference(input):
        with torch.autocast("cuda"):
            return sliding_window_inference(inputs=input, roi_size=patch_size, sw_batch_size=4, predictor=model, overlap=0.5)

    # If the fold is completely finished, skip the training loop
    if start_epoch >= max_epochs:
        print(f"[*] Fold {fold+1} already reached {max_epochs} epochs. Skipping to next fold.")
        # Make sure the best model is copied to working dir so Task 3.4 works later
        best_model_path_in = os.path.join(previous_outputs_dir, best_model_name)
        best_model_path_work = os.path.join(drive_save_dir, best_model_name)
        if not os.path.exists(best_model_path_work) and os.path.exists(best_model_path_in):
             import shutil
             shutil.copy2(best_model_path_in, best_model_path_work)
        continue

    for epoch in range(start_epoch, max_epochs):
        model.train()
        epoch_loss = 0
        step = 0
        epoch_start = time.time()

        for batch_data in train_loader:
            step += 1
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            optimizer.zero_grad()
            with torch.autocast("cuda"):
                outputs = model(inputs)
                loss = loss_function(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        epoch_loss /= step
        print(f"Fold {fold+1} | Epoch {epoch+1}/{max_epochs} | Loss: {epoch_loss:.4f} | Time: {(time.time()-epoch_start):.2f}s")

        # --- EVALUATION LOOP ---
        if (epoch + 1) % val_interval == 0:
            model.eval()
            hd95_tc_list, hd95_wt_list, hd95_et_list = [], [], []
            with torch.no_grad():
                for val_data in val_loader:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    val_outputs = inference(val_inputs)

                    val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                    val_labels_list = decollate_batch(val_labels)

                    dice_metric_batch(y_pred=val_outputs, y=val_labels_list)
                    hd95_tc_list.append(safe_hd95(val_outputs[0][0], val_labels[0][0]))
                    hd95_wt_list.append(safe_hd95(val_outputs[0][1], val_labels[0][1]))
                    hd95_et_list.append(safe_hd95(val_outputs[0][2], val_labels[0][2]))

                metric_batch = dice_metric_batch.aggregate()
                dice_tc, dice_wt, dice_et = metric_batch[0].item(), metric_batch[1].item(), metric_batch[2].item()
                mean_dice = (dice_tc + dice_wt + dice_et) / 3.0
                dice_metric_batch.reset()

                hd95_tc, hd95_wt, hd95_et = np.nanmean(hd95_tc_list), np.nanmean(hd95_wt_list), np.nanmean(hd95_et_list)

                print(f"   => Val Dice -> TC: {dice_tc:.4f}, WT: {dice_wt:.4f}, ET: {dice_et:.4f}")
                print(f"   => Val HD95 -> TC: {hd95_tc:.2f}, WT: {hd95_wt:.2f}, ET: {hd95_et:.2f}")

                if mean_dice > best_metric:
                    best_metric = mean_dice
                    save_path = os.path.join(drive_save_dir, best_model_name)
                    torch.save(model.state_dict(), save_path)
                    print(f"   [!] Saved new best model to {save_path}")
                    fold_results_dice[fold+1] = {"TC": dice_tc, "WT": dice_wt, "ET": dice_et}
                    fold_results_hd95[fold+1] = {"TC": hd95_tc, "WT": hd95_wt, "ET": hd95_et}

        # --- SAVE CHECKPOINT OVERWRITE ---
        # Overwrites the single checkpoint file for this fold at the end of every epoch.
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_metric': best_metric,
            'fold_dice': fold_results_dice.get(fold+1, {}),
            'fold_hd95': fold_results_hd95.get(fold+1, {})
        }, checkpoint_path_working)

# --- FINAL ASSIGNMENT TABLES (TASK 3.3) ---
print("\n" + "="*60)
print(f"--- 5-FOLD CROSS VALIDATION COMPLETE ---")
print(f"Total Time: {((time.time() - total_start)/60):.2f} minutes")
print("="*60)

# 1. Dice Score Summary
print(f"\n{'Fold':<10} | {'TC Dice':<10} | {'WT Dice':<10} | {'ET Dice':<10}")
print("-" * 55)
for f in sorted(fold_results_dice.keys()):
    s = fold_results_dice[f]
    if s: # Ensure data exists
        print(f"Fold {f:<5} | {s.get('TC', 0):<10.4f} | {s.get('WT', 0):<10.4f} | {s.get('ET', 0):<10.4f}")

if fold_results_dice:
    avg_tc_dice = np.mean([fold_results_dice[f].get('TC', 0) for f in fold_results_dice if fold_results_dice[f]])
    avg_wt_dice = np.mean([fold_results_dice[f].get('WT', 0) for f in fold_results_dice if fold_results_dice[f]])
    avg_et_dice = np.mean([fold_results_dice[f].get('ET', 0) for f in fold_results_dice if fold_results_dice[f]])
    print("-" * 55)
    print(f"{'AVERAGE':<10} | {avg_tc_dice:<10.4f} | {avg_wt_dice:<10.4f} | {avg_et_dice:<10.4f}")

# 2. HD95 Summary
print("\n" + "-" * 55)
print(f"{'Fold':<10} | {'TC HD95':<10} | {'WT HD95':<10} | {'ET HD95':<10}")
print("-" * 55)
for f in sorted(fold_results_hd95.keys()):
    s = fold_results_hd95[f]
    if s:
        print(f"Fold {f:<5} | {s.get('TC', 0):<10.2f} | {s.get('WT', 0):<10.2f} | {s.get('ET', 0):<10.2f}")

if fold_results_hd95:
    avg_tc_hd95 = np.nanmean([fold_results_hd95[f].get('TC', np.nan) for f in fold_results_hd95 if fold_results_hd95[f]])
    avg_wt_hd95 = np.nanmean([fold_results_hd95[f].get('WT', np.nan) for f in fold_results_hd95 if fold_results_hd95[f]])
    avg_et_hd95 = np.nanmean([fold_results_hd95[f].get('ET', np.nan) for f in fold_results_hd95 if fold_results_hd95[f]])
    print("-" * 55)
    print(f"{'AVERAGE':<10} | {avg_tc_hd95:<10.2f} | {avg_wt_hd95:<10.2f} | {avg_et_hd95:<10.2f}")
print("="*60)

In [ ]:
# --- TASK 3.4: QUALITATIVE RESULTS VISUALIZATION ---
# Load the best model from Fold 1
model.load_state_dict(torch.load(os.path.join(drive_save_dir, "best_unet_fold_1.pth")))
model.eval()

# Grab a single sample from the validation loader
with torch.no_grad():
    for val_data in val_loader:
        val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
        val_outputs = inference(val_inputs)
        val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
        break # Just need one for visualization

# Choose a slice that contains the tumor
slice_idx = 50
ground_truth = val_labels[0, 1, :, :, slice_idx].cpu().numpy() # Whole Tumor GT
prediction = val_outputs[0][1, :, :, slice_idx].cpu().numpy()  # Whole Tumor Pred

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Ground Truth (Whole Tumor)")
plt.imshow(ground_truth, cmap="gray")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Model Prediction (Whole Tumor)")
plt.imshow(prediction, cmap="gray")
plt.axis('off')

plt.suptitle("Qualitative Analysis: Ground Truth vs Prediction", fontsize=16)
plt.show()